# Predicción de precios de vivienda en California

Proyecto de Machine Learning supervisado sobre dos dominios distintos: predicción del precio medio de vivienda en California y predicción de abandono de clientes (churn) en banca. El desafío plantea dos preguntas centrales: *¿cuánto vale una vivienda dado su contexto?* y *¿qué clientes están en riesgo real de abandonar el banco?*

Este notebook recorre el pipeline completo — desde carga de datos y análisis exploratorio hasta feature engineering, entrenamiento, evaluación y diagnóstico de errores — para responder ambas preguntas con el estricto Ritual de los 7 pasos y una redacción honesta de los hallazgos: qué funciona, qué limita el modelo y cómo el propio RMSE nos mantiene humildes.


## Carga de datos

Trabajamos con dos datasets reales en formato CSV ubicados en `data/raw/`:

- **California Housing Prices**: precios de vivienda en California (1990) a nivel de bloque censal, con el target `median_house_value` en **dólares reales**.
- **Bank Customer Churn**: 10.000 clientes de un banco europeo con su indicador de abandono (`Exited`).

Ambos se cargan tal cual, sin modificaciones sobre los archivos originales.


In [ ]:
#  Imports y Configuración 

# Importamos las librerías de análisis y visualización de datos.
import pandas as pd          
import numpy as np           
import matplotlib.pyplot as plt  
import seaborn as sns        

# 1. División de Datos y Optimización (sklearn.model_selection) 
from sklearn.model_selection import train_test_split # train_test_split: division de datos en train y test (el modelo aprende con train y se mide con test, como en el reto).

# 2. Limpieza y Transformación de Datos  (sklearn.preprocessing e impute) 
from sklearn.preprocessing import StandardScaler, OneHotEncoder
#   StandardScaler: escala los números para que tengan media 0 y desvía estándar 1.
#     Crucial para Regresión Logística o K-Means: evita que una variable con números
#     gigantes (salario) opaque a una con números pequeños (edad).
#   OneHotEncoder: convierte texto/categorías (“Rojo”, “Verde”) en columnas de 0/1.
#     Los modelos sólo entienden números, así que este paso es obligatorio.
from sklearn.compose import ColumnTransformer
#   ColumnTransformer: aplica transformaciones DIFERENTES a columnas DIFERENTES en un
#     solo paso (ej: StandardScaler a numéricas + OneHotEncoder a texto a la vez).
from sklearn.pipeline import Pipeline
#   Pipeline: “fábrica” o cadena de montaje. Pega preprocesamiento + modelo en un solo
#     objeto; con .fit() limpia, transforma y entrena automáticamente y en orden.
from sklearn.impute import SimpleImputer
#   SimpleImputer: llena los datos faltantes (nulos/NaN) con el promedio, la mediana
#     o el valor más repetido de la columna (usamos mediana para total_bedrooms).
from sklearn.cluster import KMeans
#   KMeans: aprendizaje NO supervisado. Agrupa datos por similitud sin etiquetas
#     previas (creamos ZONAS GEOGRÁFICAS a partir de latitud/longitud).


# 3. Modelos de Machine Learning (Algoritmos)
from sklearn.linear_model import LinearRegression, LogisticRegression
#   LinearRegression: regresión clásica. Predice un valor continuo (precio de una casa)
#     dibujando la línea que mejor se ajusta (mínimos cuadrados / OLS).
#   LogisticRegression: para CLASIFICACIÓN binaria (predecir si ocurre un evento o no,
#     como el Churn o el Spam). Devuelve la probabilidad de pertenecer a la clase 1.


# 4. Metricas de Evaluacion de Modelos (sklearn.metrics)
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, classification_report,
                             confusion_matrix)
from sklearn.metrics import ConfusionMatrixDisplay
#   --- Para REGRESIÓN (predicción de números) ---
#   mean_squared_error: calcula el MSE (error cuadrático medio). Con raíz cuadrada da
#     el RMSE, el error promedio en las unidades originales ($).
#   r2_score (R²): qué porcentaje de la variación de los datos explica el modelo
#     (0 a 1, donde 1 = ajuste perfecto). Lo usamos para California.
#   --- Para CLASIFICACIÓN (predicción de categorías) ---
#   accuracy_score: % total de predicciones correctas (a cuántos acerté en total).
#   precision_score: de los clasificados como “Positivos”, cuántos lo eran realmente
#     (de los que predije que harían Churn, cuántos sí lo hicieron). Evita falsos positivos.
#   recall_score: sensibilidad. De los que realmente hicieron Churn, cuántos detectó.
#     Evita falsos negativos.
#   classification_report: resumen completo (accuracy, precision, recall, f1 por clase).
#   confusion_matrix: cruza valores reales vs predicciones (aciertos, falsos pos./neg.).
#   ConfusionMatrixDisplay: grafica la matriz de confusión con colores para leerla fácil.


# c) Configuracion general
import warnings
warnings.filterwarnings('ignore')          
sns.set_theme(style='whitegrid')          

SEED = 42                                  # semilla fija: misma partición en cada ejecución (reproducibilidad)

# Rutas relativas al notebook (data/raw/) y carga de ambos datasets
# Churn: dataset clásico de 10.000 clientes (abbas829/bank-customer-churn -> Bank_Churn.csv).
CAL = r'data/raw/housing.csv'              # dataset de regresión (California)
BNK = r'data/raw/Bank_Churn.csv'           # dataset de clasificación (Bank churn)

df_cal_raw = pd.read_csv(CAL)             
df_bnk_raw = pd.read_csv(BNK)              

print('California Housing:', df_cal_raw.shape[0], 'filas x', df_cal_raw.shape[1], 'columnas')
print('Bank Churn      :', df_bnk_raw.shape[0], 'filas x', df_bnk_raw.shape[1], 'columnas')


## Contexto: hallazgos clave del EDA

Antes de modelar, miremos los datos a la cara. Las decisiones de preparación se basan en lo que el análisis exploratorio revela, no en suposiciones.

### California Housing

- `total_bedrooms` tiene **207 nulos** que deben imputarse.
- `median_income` es el predictor más fuerte del precio (correlación ~0.69 en la serie completa).
- `ocean_proximity` es categórica (`<1H OCEAN`, `INLAND`, `ISLAND`, `NEAR BAY`, `NEAR OCEAN`) y debe codificarse.
- El target tiene un **techo artificial de $500,001**: las viviendas de lujo fueron recortadas a ese valor, lo que distorsiona el error. Se eliminan en el feature engineering.
- `latitude` y `longitude` capturan ubicación pero como coordenadas crudas; las convertiremos en **zonas geográficas con K-Means** para hacerlas interpretables.
- `total_rooms` y `total_bedrooms` están correlacionadas entre sí (multicolinealidad).

### Bank Churn

- El dataset está **libre de nulos** (10.000 filas de clientes de un banco europeo).
- El target `Exited` está **desbalanceado** (79.63% se queda / 20.37% se va) — el baseline de Accuracy es ~79.63%, y el **Recall será clave** para detectar fugas.
- `Age` es el predictor más fuerte del churn: los clientes que se van tienen una edad media de **44.8 años** vs 37.4 de los que se quedan.
- `Geography` revela un patrón regional: **Alemania** concentra el 32% de las bajas, vs ~16% en Francia y España — `Geography_Germany` será un predictor relevante.
- `IsActiveMember` tiene un **efecto protector**: los miembros activos abandonan menos (promedio 0.64 en se queda vs 0.36 en se va).
- `NumOfProducts` tiene una relación **no lineal** con el churn (más de 2 productos se asocia a mayor fuga); se conserva numérico para que el modelo capture la señal dominante.


In [ ]:
# eda rapido: california housing

print('\n========== EDA RÁPIDO: CALIFORNIA HOUSING ==========')
print(f'Dimensiones: {df_cal_raw.shape}')

print(f'\nNulos por columna:\n{df_cal_raw.isnull().sum()}')

# los modelos solo entienden números -> esto se tendrá que codificar (one-hot).
print('\nValores únicos en ocean_proximity:')
print(df_cal_raw['ocean_proximity'].value_counts())


In [ ]:
# eda rapido: bank churn

import pandas as pd
from IPython.display import display

print('\n========== EDA RÁPIDO: BANK CHURN ==========')
print('Dimensiones:', df_bnk_raw.shape)

# Se normalizan los datos a proporciones (0 a 1)
print('\nDistribución del target Exited:')
print(df_bnk_raw['Exited'].value_counts(normalize=True).round(4).to_string())

# Bajas (tasa) y clientes por país en tabla
baja_pais = df_bnk_raw.groupby('Geography')['Exited'].agg(['count','mean']).rename(columns={'count':'Clientes','mean':'Tasa de bajas'}).round(4)
print('\n--- Distribucion y Bajas por país (lado a lado) ---')
print(baja_pais)

# Edad media y miembros activos según Exited
edad_active = df_bnk_raw.groupby('Exited').agg(Edad_media=('Age','mean'), Miembros_activos=('IsActiveMember','mean')).round(3)
edad_active.index = ['Se queda (0)','Se va (1)']
print('\n--- Edad y actividad según Exited (lado a lado) ---')
print(edad_active)


## Feature engineering

Con los datos explorados, el siguiente paso es transformarlos en algo que el modelo pueda aprovechar mejor. Creamos nuevas variables a partir de las originales — densidades con sentido económico, interacciones y zonas geográficas — para darle al modelo mejor materia prima sin agregar información externa.


In [ ]:
# feature engineering: california
# Tres ideas del desafío: quitar el techo artificial del target, derivar ratios con sentido económico y crear ZONAS GEOGRÁFICAS con K-Means.

# Trabajamos sobre una copia para no tocar el DataFrame crudo.
df_cal = df_cal_raw.copy()

# El Techo artificial distorsiona el rmse por eso hay que eliminarlo
print('Filas con target == 500001 (techo):', (df_cal['median_house_value'] >= 500001).sum()) 
df_cal = df_cal[df_cal['median_house_value'] < 500001].reset_index(drop=True) # reset_index(drop=True) renumeran las filas de 0 en adelante tras filtrar.
print('Filas tras eliminar el techo:', len(df_cal))

# Ratios
df_cal['rooms_per_household'] = df_cal['total_rooms'] / df_cal['households']
df_cal['population_per_room'] = df_cal['population'] / df_cal['total_rooms']

# concepto del challenge (interacción / cross feature): (crear una variable nueva combinando dos variables existentes para capturar un efecto sinérgico)
# 3) age_x_income: multiplicar antigüedad x ingreso captura un EFECTO COMBINADO que ninguna variable ve por separado (una casa vieja y rica no equivale a vieja + rica).
df_cal['age_x_income'] = df_cal['housing_median_age'] * df_cal['median_income']  # El valor numérico de esta nueva columna se dispara únicamente cuando ambos valores son altos al mismo tiempo.

# concepto del challenge (aprendizaje NO supervisado dentro de una misión supervisada):
# 4) Zonas geográficas con K-Means. En lugar de pasar lat/long crudas (poco interpretables), agrupamos las ubicaciones en 8 zonas y usamos la zona como variable categórica.
#    importante: no usamos el target para agrupar -> NO hay fuga de datos.
geo = df_cal[['latitude', 'longitude']].values   # matriz de coordenadas

# Instanciamos K-Means con 8 clusters fijos por la semilla (reproducibilidad).
# n_init=10 repite el algoritmo 10 veces y se queda con el mejor agrupamiento.
kmeans = KMeans(n_clusters=8, random_state=SEED, n_init=10) #n_clusters= cantidad de grupos, n_init = cuantas veces hara el proceso de agrupacion

df_cal['geo_zone'] = kmeans.fit_predict(geo) #Calcula las distancias y agrupa todos tus distritos en 8 regiones.

# get_dummies: convierte la zona en columnas 0/1. drop_first evita redundancia (k-1 dummies).
geo = pd.get_dummies(df_cal['geo_zone'], prefix='geo_zone').astype(int)

# Concatenamos las dummies geográficas al dataset.
df_cal = pd.concat([df_cal, geo], axis=1)

# Nota: los nulos de total_bedrooms NO se imputan aquí a propósito;
#       los imputaremos dentro del Pipeline (paso siguiente) para no filtrar datos.
print('\nRatios derivados (muestra):')
print(df_cal[['rooms_per_household', 'population_per_room', 'age_x_income']].head(3))
print('\nZonas geográficas creadas:', list(geo.columns)[:5], '... total', len(geo.columns))
